# NB4 — The panel, the pre-trends, and the power
### The Peltzman Protocol · MSR 2027

This is the gate. Three questions, in order, and the third one decides whether the study proceeds as designed.

1. **Is the panel honest?** NB3 emitted a row only where something happened, so quiet months are absent rather than zero. Left alone, every estimate would silently condition on activity — which treatment itself affects.
2. **Are the pre-trends flat?** If review effort was already falling before projects adopted gating, the gate did not cause the fall and the paper becomes descriptive. Better to learn that today than after fifteen hours of estimation.
3. **How small an effect can this design see?** Not from a formula with a guessed standard deviation — by randomisation. Assign fake adoption dates to never-treated repos, estimate the effect, repeat. The spread of those estimates *is* the standard error of this design at this sample size, and their centre is a placebo test that should sit on zero.

The prior study found roughly one fewer comment per pull request. Whatever the minimum detectable effect turns out to be, it has to be comfortably below that for a null anywhere to mean anything.


## 0 · Setup

In [ ]:
#@title Configuration { display-mode: "form" }
WINDOW_START = "2019-07"  #@param {type:"string"}
WINDOW_END   = "2025-06"  #@param {type:"string"}
MIN_SPAN     = 24         #@param {type:"integer"}
MIN_PRS      = 1          #@param {type:"integer"}
N_PERM       = 200        #@param {type:"integer"}
LEADS        = 12         #@param {type:"integer"}
LAGS         = 12         #@param {type:"integer"}
SEED         = 20261005   #@param {type:"integer"}

import os, sys, json, time, gzip, csv, subprocess
from pathlib import Path
import numpy as np, pandas as pd
IN_COLAB=False
try:
    from google.colab import drive; IN_COLAB=True
except ImportError: pass
DRIVE_OK=False
if IN_COLAB:
    try: drive.mount("/content/drive",force_remount=False); DRIVE_OK=True
    except Exception as e: print("!! Drive mount FAILED:",e)
DRIVE=("/content/drive/MyDrive/peltzman" if DRIVE_OK else
       "/content/peltzman_LOCAL" if IN_COLAB else os.path.expanduser("~/peltzman_out"))
STATE=Path(DRIVE,"state"); RAW=Path(DRIVE,"raw"); CODE=Path("/content" if IN_COLAB else "/tmp")
FIG=Path(DRIVE,"figures"); FIG.mkdir(parents=True,exist_ok=True)
for _p in (STATE,CODE): Path(_p).mkdir(parents=True,exist_ok=True)
sys.path.insert(0,str(CODE))
print("state:",STATE)

In [ ]:
%%writefile /content/peltzman_io.py
"""
Peltzman Protocol - durable, resumable job state.

Guarantee
---------
Kill the process, lose the runtime, close the browser, at ANY instant:
  * no file is ever left half-written,
  * no completed work is ever lost,
  * no work is ever silently duplicated,
  * restarting the cell resumes from the last completed batch.

How
---
The key list is fixed and ordered at job creation, and batch `i` is defined as
a deterministic slice of it. A batch's output is therefore a pure function of
its index, so re-running a batch overwrites its own file and can never
duplicate rows. Completion is represented by the *existence* of that file and
nothing else - there is no separate done-list that can drift out of sync with
the data.

Every file is written to a temporary name, flushed, fsync'd, and then moved
into place with os.replace(), which is atomic. A crash mid-write leaves a
stray .tmp file (harmless, cleaned on next run) and the previous state intact.

Because completion is file existence, the crash window is exactly one batch.
Size batches so one takes a few minutes.
"""
from __future__ import annotations
import json, os, time, hashlib, tempfile, shutil, glob, sys
from pathlib import Path

__all__ = ["Job", "atomic_write_text", "atomic_write_bytes"]


# --------------------------------------------------------------- atomic io --
def atomic_write_bytes(path, data: bytes):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=str(path.parent), prefix=path.name + ".", suffix=".tmp")
    try:
        with os.fdopen(fd, "wb") as f:
            f.write(data)
            f.flush()
            os.fsync(f.fileno())          # no-op on some mounts; harmless
        os.replace(tmp, path)             # atomic within a filesystem
    except BaseException:
        try: os.unlink(tmp)
        except OSError: pass
        raise


def atomic_write_text(path, text: str):
    atomic_write_bytes(path, text.encode("utf-8"))


def _hash_params(params) -> str:
    blob = json.dumps(params, sort_keys=True, default=str).encode()
    return hashlib.sha1(blob).hexdigest()[:10]


# --------------------------------------------------------------------- job --
class Job:
    """A resumable pass over an ordered list of keys.

    >>> job = Job("nb1_workflows", keys=repos, root=DRIVE, batch_size=25,
    ...           params={"detector": "v3"})
    >>> for i, keys in job.pending():
    ...     job.commit(i, [row for k in keys for row in scan(k)])
    >>> df = job.frame()

    Changing `params` starts a new, separate run directory: results computed
    under different settings are never silently mixed.
    """

    def __init__(self, name, keys, root, batch_size=25, params=None, verbose=True):
        self.name = name
        self.params = dict(params or {})
        self.batch_size = int(batch_size)
        self.verbose = verbose
        # NOTE: the key count is deliberately NOT part of the run identity.
        # If the input list changes at all we want a loud error, not a silent
        # fresh run that looks like "nothing was ever done".
        self.dir = Path(root) / name / _hash_params(
            {"params": self.params, "batch_size": self.batch_size})
        self.shards = self.dir / "shards"
        self.shards.mkdir(parents=True, exist_ok=True)

        keyfile = self.dir / "keys.json"
        if keyfile.exists():
            stored = json.loads(keyfile.read_text())
            if stored != list(keys):
                raise ValueError(
                    f"{self.name}: key list changed since this run started "
                    f"({len(stored)} -> {len(keys)}). Change `params` to start "
                    f"a new run, or restore the original list.")
        else:
            atomic_write_text(keyfile, json.dumps(list(keys)))
            atomic_write_text(self.dir / "meta.json", json.dumps({
                "name": name, "created": time.time(), "n_keys": len(keys),
                "batch_size": self.batch_size, "params": self.params,
                "python": sys.version.split()[0]}, indent=1))

        self.keys = list(keys)
        self._sweep_tmp()
        if verbose:
            d, n = self.n_done_batches(), self.n_batches()
            print(f"[{name}] {self.dir}")
            print(f"[{name}] {d}/{n} batches complete"
                  + (f" — resuming at batch {d}" if 0 < d < n else
                     " — nothing to do" if d == n else " — starting fresh"))

    # ---------------------------------------------------------- internals --
    def _sweep_tmp(self):
        """Remove leftovers from a crash mid-write. They are never valid."""
        for t in glob.glob(str(self.shards / "*.tmp")) + glob.glob(str(self.dir / "*.tmp")):
            try: os.unlink(t)
            except OSError: pass

    def _path(self, i) -> Path:
        return self.shards / f"batch-{i:06d}.jsonl"

    def n_batches(self) -> int:
        return (len(self.keys) + self.batch_size - 1) // self.batch_size

    def batch_keys(self, i):
        return self.keys[i * self.batch_size:(i + 1) * self.batch_size]

    def done_batches(self):
        return {i for i in range(self.n_batches()) if self._path(i).exists()}

    def n_done_batches(self) -> int:
        return len(self.done_batches())

    # ------------------------------------------------------------- driving --
    def pending(self):
        """Yield (batch_index, keys) for every batch not yet committed."""
        done = self.done_batches()
        for i in range(self.n_batches()):
            if i not in done:
                yield i, self.batch_keys(i)

    def commit(self, i, rows):
        """Atomically record this batch's output. Idempotent by construction."""
        payload = "".join(json.dumps(r, separators=(",", ":"), default=str) + "\n"
                          for r in rows)
        atomic_write_text(self._path(i), payload)
        self.beat(i, len(rows))

    def beat(self, i, n_rows=None, note=None):
        atomic_write_text(self.dir / "heartbeat.json", json.dumps({
            "at": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
            "last_batch": i, "rows": n_rows, "note": note,
            "done": self.n_done_batches(), "total": self.n_batches()}, indent=1))

    def run(self, fn, on_error="record", progress_every=1, after_batch=None):
        """Drive the whole job. `fn(key) -> list[dict]`.

        on_error='record'  capture the failure as a row and move on (default)
                 'skip'    drop the key silently
                 'raise'   stop, keeping every committed batch
        after_batch(job, i)  optional hook run after each successful commit,
                 e.g. to replicate state off the runtime. A failing hook never
                 loses the batch - it is already on disk before the hook runs.
        """
        t0, n_before = time.time(), self.n_done_batches()
        for i, keys in self.pending():
            rows = []
            for k in keys:
                try:
                    rows.extend(fn(k) or [])
                except KeyboardInterrupt:
                    raise
                except Exception as e:
                    if on_error == "raise":
                        raise
                    if on_error == "record":
                        rows.append({"_key": k, "_error": f"{type(e).__name__}: {e}"})
            self.commit(i, rows)
            if after_batch is not None:
                try: after_batch(self, i)
                except Exception as e:
                    print(f"  after_batch hook failed ({type(e).__name__}: {e}) — "
                          f"batch is still committed locally")
            done = self.n_done_batches()
            if self.verbose and (done - n_before) % progress_every == 0:
                el = time.time() - t0
                per = el / max(1, done - n_before)
                left = (self.n_batches() - done) * per
                print(f"  batch {done}/{self.n_batches()}  "
                      f"{per:5.1f}s/batch  ~{left/60:5.1f} min remaining")
        if self.verbose:
            print(f"[{self.name}] complete: {self.n_batches()} batches")

    # ------------------------------------------------------------ reading --
    def rows(self, drop_errors=True):
        """Iterate every committed row. Truncated trailing lines - which an
        atomic write cannot produce, but a copied/synced file might - are
        skipped and counted rather than raising."""
        bad = 0
        for i in sorted(self.done_batches()):
            with open(self._path(i), "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        bad += 1
                        continue
                    if drop_errors and isinstance(r, dict) and "_error" in r:
                        continue
                    yield r
        if bad and self.verbose:
            print(f"[{self.name}] warning: skipped {bad} unparseable line(s) — "
                  f"run .verify() to locate")

    def frame(self, drop_errors=True):
        import pandas as pd
        return pd.DataFrame(list(self.rows(drop_errors=drop_errors)))

    def errors(self):
        return [r for r in self.rows(drop_errors=False)
                if isinstance(r, dict) and "_error" in r]

    # -------------------------------------------------------- maintenance --
    def verify(self):
        """Check every committed shard parses. Returns list of bad batch ids."""
        bad = []
        for i in sorted(self.done_batches()):
            try:
                for line in open(self._path(i), encoding="utf-8"):
                    if line.strip():
                        json.loads(line)
            except Exception as e:
                bad.append((i, f"{type(e).__name__}: {e}"))
        if self.verbose:
            print(f"[{self.name}] verify: {self.n_done_batches()} shards, "
                  f"{len(bad)} damaged")
        return bad

    def reset(self, batches=None):
        """Delete shards so they are recomputed. `batches=None` resets damaged
        ones only; pass an iterable of indices, or 'all'."""
        if batches == "all":
            targets = sorted(self.done_batches())
        elif batches is None:
            targets = [i for i, _ in self.verify()]
        else:
            targets = list(batches)
        for i in targets:
            try: os.unlink(self._path(i))
            except OSError: pass
        if self.verbose:
            print(f"[{self.name}] reset {len(targets)} batch(es)")
        return targets

    def retry_errors(self):
        """Re-queue every batch that recorded at least one failure."""
        hit = set()
        for i in sorted(self.done_batches()):
            for line in open(self._path(i), encoding="utf-8"):
                if '"_error"' in line:
                    hit.add(i); break
        return self.reset(hit)

    def export(self, path, fmt="parquet"):
        """Write the whole result set as one file, atomically."""
        df = self.frame()
        p = Path(path); p.parent.mkdir(parents=True, exist_ok=True)
        tmp = p.with_suffix(p.suffix + ".tmp")
        if fmt == "parquet":
            df.to_parquet(tmp, index=False)
        else:
            df.to_csv(tmp, index=False)
        os.replace(tmp, p)
        if self.verbose:
            print(f"[{self.name}] exported {len(df):,} rows -> {p}")
        return p

    def status(self):
        hb = self.dir / "heartbeat.json"
        return {"dir": str(self.dir), "batches": self.n_batches(),
                "done": self.n_done_batches(), "keys": len(self.keys),
                "heartbeat": json.loads(hb.read_text()) if hb.exists() else None}


In [ ]:
%%writefile /content/panel_tools.py
"""
Peltzman Protocol - panel construction, fast two-way fixed-effects DiD, and
randomisation-based power.

The estimator here is deliberately simple: it exists to check pre-trends and to
measure how much noise the design has, not to produce the paper's numbers.
Callaway-Sant'Anna in NB5 does that. But a fast TWFE fit is what makes a
permutation study affordable, and a permutation study is a far more honest
account of power than plugging a guessed standard deviation into a formula.
"""
from __future__ import annotations
import numpy as np

__all__ = ["month_index", "demean_two_way", "twfe_did", "event_study",
           "placebo_se", "mde_from_se"]


def month_index(ym):
    """'YYYY-MM' -> integer month, so arithmetic and sorting are trivial."""
    y, m = str(ym).split("-")
    return int(y) * 12 + int(m) - 1


def demean_two_way(y, unit, time, iters=30, tol=1e-10):
    """Absorb unit and time fixed effects by alternating projections.

    Equivalent to residualising on the full dummy set, but O(n) per sweep
    instead of building a matrix with tens of thousands of columns. Converges
    geometrically; 30 sweeps is far past what a panel this shape needs.
    """
    y = np.asarray(y, dtype=np.float64).copy()
    nu, nt = unit.max() + 1, time.max() + 1
    cu, ct = np.bincount(unit, minlength=nu), np.bincount(time, minlength=nt)
    cu[cu == 0] = 1; ct[ct == 0] = 1
    for _ in range(iters):
        prev = y[:1000].copy() if y.size >= 1000 else y.copy()
        y -= (np.bincount(unit, y, minlength=nu) / cu)[unit]
        y -= (np.bincount(time, y, minlength=nt) / ct)[time]
        cur = y[:1000] if y.size >= 1000 else y
        if np.max(np.abs(cur - prev)) < tol:
            break
    return y


def twfe_did(y, treat, unit, time, cluster=None):
    """Two-way fixed-effects DiD. Returns (beta, se, n).

    `treat` is the post-treatment indicator. Standard errors are clustered on
    `cluster` (default: unit), because repo-months within a repo are anything
    but independent and unclustered errors here would be fiction.
    """
    y = np.asarray(y, float); d = np.asarray(treat, float)
    keep = np.isfinite(y) & np.isfinite(d)
    y, d, unit, time = y[keep], d[keep], unit[keep], time[keep]
    if y.size < 10 or d.sum() < 2 or (1 - d).sum() < 2:
        return np.nan, np.nan, int(y.size)
    yt = demean_two_way(y, unit, time)
    dt = demean_two_way(d, unit, time)
    denom = float(dt @ dt)
    if denom <= 0:
        return np.nan, np.nan, int(y.size)
    beta = float(dt @ yt) / denom
    resid = yt - beta * dt
    g = unit if cluster is None else np.asarray(cluster)[keep]
    order = np.argsort(g, kind="stable")
    gs, sc, rs = g[order], dt[order], resid[order]
    bounds = np.flatnonzero(np.r_[True, gs[1:] != gs[:-1], True])
    meat = 0.0
    for a, b in zip(bounds[:-1], bounds[1:]):
        meat += float(sc[a:b] @ rs[a:b]) ** 2
    ng = len(bounds) - 1
    if ng < 2:
        return beta, np.nan, int(y.size)
    adj = ng / (ng - 1.0)
    return beta, float(np.sqrt(adj * meat) / denom), int(y.size)


def event_study(y, evt, unit, time, leads=12, lags=12, ref=-1):
    """Coefficients by event time. `evt` is months since treatment; use a large
    sentinel (e.g. 9999) for never-treated units so they act as controls.

    Returns {event_time: (beta, se)} with `ref` omitted as the base period.
    """
    y = np.asarray(y, float); evt = np.asarray(evt)
    out = {}
    for k in range(-leads, lags + 1):
        if k == ref:
            continue
        d = (evt == k).astype(float)
        # compare period k against the reference period and the never-treated
        mask = np.isfinite(y) & ((evt == k) | (evt == ref) | (evt > 9000))
        if d[mask].sum() < 5:
            out[k] = (np.nan, np.nan); continue
        b, se, _ = twfe_did(y[mask], d[mask], unit[mask], time[mask])
        out[k] = (b, se)
    return out


def placebo_se(y, unit, time, n_treated, cohort_months, rng, n_perm=200,
               min_post=6):
    """Randomisation inference on never-treated units.

    Assign `n_treated` units a fake adoption month drawn from the real cohort
    distribution, estimate the DiD, repeat. The spread of the resulting
    estimates IS the standard error of this design at this sample size - no
    distributional assumption, no guessed sigma. The centre doubles as a
    placebo test: it should sit on zero.
    """
    y = np.asarray(y, float)
    units = np.unique(unit)
    if len(units) <= n_treated or len(cohort_months) == 0:
        return np.nan, np.nan, 0
    tmax = time.max()
    est = []
    for _ in range(n_perm):
        chosen = rng.choice(units, size=n_treated, replace=False)
        gmap = {u: rng.choice(cohort_months) for u in chosen}
        d = np.zeros(y.size)
        sel = np.isin(unit, chosen)
        gs = np.array([gmap.get(u, 10**9) for u in unit[sel]])
        d[sel] = (time[sel] >= gs).astype(float)
        if d.sum() < min_post:
            continue
        b, _, _ = twfe_did(y, d, unit, time)
        if np.isfinite(b):
            est.append(b)
    if len(est) < 20:
        return np.nan, np.nan, len(est)
    est = np.array(est)
    return float(est.mean()), float(est.std(ddof=1)), len(est)


def mde_from_se(se, alpha=0.05, power=0.80):
    """Smallest effect detectable at the given power. 1.96 + 0.84 for the
    conventional 5% / 80%."""
    return np.nan if not np.isfinite(se) else float(2.802 * se)


In [ ]:
import importlib, peltzman_io, panel_tools
importlib.reload(peltzman_io); importlib.reload(panel_tools)
from peltzman_io import atomic_write_text
from panel_tools import (month_index, demean_two_way, twfe_did, event_study,
                         placebo_se, mde_from_se)

# self-check on synthetic data with a known effect - if this fails, nothing below means anything
_rng=np.random.default_rng(0); NU,NT,TRUE=400,48,-0.5
_u=np.repeat(np.arange(NU),NT); _t=np.tile(np.arange(NT),NU)
_g={u:_rng.integers(12,36) for u in _rng.choice(NU,150,replace=False)}
_gs=np.array([_g.get(u,10**9) for u in _u]); _p=(_t>=_gs).astype(float)
_y=_rng.normal(0,2,NU)[_u]+TRUE*_p+_rng.normal(0,1,NU*NT)
_b,_se,_=twfe_did(_y,_p,_u,_t)
assert abs(_b-TRUE)<3*_se, f"estimator self-check failed: {_b:.3f} vs {TRUE}"
print(f"estimator self-check OK: recovered {_b:+.3f} (true {TRUE:+.2f}, se {_se:.3f})")

## 1 · Load and balance

A repo-month with no events is a **zero**, not a missing observation. But a repo that did not exist yet, or has been dead for years, is genuinely outside the panel. The rule: fill zeros from the later of the window start and the repository's creation, through the last month it showed any activity. That treats silence inside a project's life as the data it is, without inventing a decade of zeros for an abandoned project.

In [ ]:
panel = pd.read_parquet(Path(STATE,"nb3_panel.parquet"))
tre   = pd.read_parquet(Path(STATE,"nb1_treatments.parquet"))
never = set(json.loads(Path(STATE,"nb1_controls.json").read_text()))
cfg   = (pd.read_parquet(Path(STATE,"nb2_config_treatments.parquet"))
         if Path(STATE,"nb2_config_treatments.parquet").exists() else pd.DataFrame())
print(f"NB3 panel {len(panel):,} rows · {panel.repo.nunique():,} repos")

created={}
with gzip.open(RAW/"repositories.csv.gz","rt",errors="replace") as f:
    for row in csv.DictReader(f):
        created[row["name"]]=(row.get("created") or "")[:7]

W0,W1 = month_index(WINDOW_START), month_index(WINDOW_END)
panel["t"] = panel.month.map(month_index)
span = panel.groupby("repo").t.agg(["min","max"])
born = pd.Series({r: month_index(c) if c else W0 for r,c in created.items()})
span["lo"] = np.maximum(span.index.map(lambda r: born.get(r, W0)).values, W0)
span["lo"] = np.maximum(span["lo"], span["min"])   # never before first observed activity
span["hi"] = np.minimum(span["max"], W1)
span = span[(span.hi - span.lo + 1) >= MIN_SPAN]
print(f"{len(span):,} repos with >= {MIN_SPAN} months of plausible life")

grid = pd.DataFrame({"repo": np.repeat(span.index.values,
                                       (span.hi-span.lo+1).values),
                     "t": np.concatenate([np.arange(a,b+1) for a,b in
                                          zip(span.lo.values, span.hi.values)])})
bal = grid.merge(panel.drop(columns=["month"]), on=["repo","t"], how="left")
count_cols=[c for c in panel.columns if c not in ("repo","month","t","med_lat_h","mean_lat_h")]
bal[count_cols]=bal[count_cols].fillna(0)
print(f"balanced panel {len(bal):,} rows ({len(bal)/max(1,len(panel)):.2f}x the raw panel) · "
      f"{bal.repo.nunique():,} repos")
print(f"filled zeros: {(len(bal)-len(panel))/len(bal):.1%} of rows")

## 2 · Outcomes

Rates need a denominator, and a rate computed on one pull request is noise. Every ratio is therefore defined only where the denominator clears `MIN_PRS`; elsewhere it is missing and the estimator drops it. Counts are kept alongside in log form as a robustness family — if the two disagree, that is worth knowing before the paper claims anything.

In [ ]:
d = bal
# Denominators use HUMAN pull requests. Bot PRs are the confound that made
# Dependabot look like it halved reviewer attention: adopting a bot inflates
# prs_opened with changes nobody reviews, so any per-PR ratio falls
# mechanically. Raw columns stay in the panel for the sensitivity table.
H = "prs_opened_h" in d.columns
den_pr  = (d.prs_opened_h if H else d.prs_opened).where(
           (d.prs_opened_h if H else d.prs_opened) >= MIN_PRS)
den_mrg = (d.prs_merged_h if H else d.prs_merged).where(
           (d.prs_merged_h if H else d.prs_merged) >= MIN_PRS)
den_att = (d.merged_attributable_h if H else d.merged_attributable).where(
           (d.merged_attributable_h if H else d.merged_attributable) >= MIN_PRS)
print("denominators:", "human-only" if H else "ALL ACTORS — run NB3b first")

col = lambda base: d[base+"_h"] if H and (base+"_h") in d.columns else d[base]
d["o1_reviewers_per_pr"]   = col("reviewers")       / den_pr
d["o2_comments_per_pr"]    = col("review_comments") / den_pr
d["o2b_reviews_per_pr"]    = col("reviews")         / den_pr
d["o3_review_coverage"]    = (col("prs_reviewed")   / den_pr).clip(upper=1.0)
d["o3b_changes_req_rate"]  = d.changes_req          / den_pr
# log: raw churn is so heavy-tailed that the merge-queue placebo centred at -300
d["o4_churn_per_merged"]   = np.log1p(col("churn_merged") / den_mrg)
d["o5_self_merge_rate"]    = col("self_merged")     / den_att
d["o6_direct_push_share"]  = col("pushes_default")  / (col("pushes_default") + den_mrg)
for c in ("reviewers","review_comments","reviews","prs_opened","pushes_default"):
    d["log_"+c] = np.log1p(col(c))

OUTCOMES=[c for c in d.columns if c.startswith(("o1_","o2_","o2b","o3_","o3b","o4_","o5_","o6_"))]
print("\noutcome coverage (share of repo-months where it is defined)\n")
for c in OUTCOMES:
    v=d[c]
    print(f"  {c:24s} {v.notna().mean():6.1%}  mean {v.mean():9.3f}  sd {v.std():9.3f}  "
          f"median {v.median():8.3f}")

## 3 · Attach treatments and event time

Never-treated repos get a sentinel cohort so the estimator uses them as controls rather than dropping them.

In [ ]:
NEVER = 10**9
A = tre[tre.treatment=="A_first_gating"].set_index("repo").month.map(month_index)
gmapA = d.repo.map(A).fillna(NEVER)
d["g_A"] = gmapA.values
d["evt_A"] = np.where(d.g_A>=NEVER, 9999, d.t - d.g_A)
d["post_A"] = (d.t >= d.g_A).astype(float)

B = {t: sub.set_index("repo").month.map(month_index)
     for t, sub in tre[tre.treatment!="A_first_gating"].groupby("treatment")}
for t, s in B.items():
    g = d.repo.map(s).fillna(NEVER)
    d[f"g_{t}"] = g.values
    d[f"evt_{t}"] = np.where(g>=NEVER, 9999, d.t - g)
    d[f"post_{t}"] = (d.t >= g).astype(float)
if len(cfg):
    for t, sub in cfg.groupby("treatment"):
        s = sub.set_index("repo").month.map(month_index)
        g = d.repo.map(s).fillna(NEVER)
        d[f"g_{t}"] = g.values
        d[f"evt_{t}"] = np.where(g>=NEVER, 9999, d.t - g)
        d[f"post_{t}"] = (d.t >= g).astype(float)

TREATS=[c[2:] for c in d.columns if c.startswith("g_")]
print("treated repos present in the balanced panel")
for t in TREATS:
    n=int((d[f"g_{t}"]<NEVER).groupby(d.repo).max().sum())
    print(f"  {t:18s} {n:6,}")
uid = {r:i for i,r in enumerate(d.repo.unique())}
d["u"] = d.repo.map(uid).astype(np.int64)
d["tt"] = (d.t - d.t.min()).astype(np.int64)

## 4 · Pre-trends — the gate

Leads −12 to −1 against the reference period. If they slope, adoption followed the change rather than causing it.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

PRIMARY = "o1_reviewers_per_pr"
def run_es(outcome, treat, leads=LEADS, lags=LAGS):
    m = d[outcome].notna().values
    return event_study(d[outcome].values[m], d[f"evt_{treat}"].values[m],
                       d.u.values[m], d.tt.values[m], leads=leads, lags=lags)

es_A = run_es(PRIMARY, "A")
ks = sorted(es_A); bs=[es_A[k][0] for k in ks]; ses=[es_A[k][1] for k in ks]
pre = [(k,b,s) for k,b,s in zip(ks,bs,ses) if k<0 and np.isfinite(b)]
zs  = [b/s for _,b,s in pre if s and np.isfinite(s)]
chi2 = float(np.sum(np.square(zs)))
from math import erfc, sqrt
print(f"pre-period leads: {len(pre)} · mean {np.mean([b for _,b,_ in pre]):+.4f} · "
      f"max |z| {max(abs(z) for z in zs):.2f} · sum z^2 = {chi2:.1f} on {len(zs)} df")
print("VERDICT:", "pre-trends look flat — proceed"
      if max(abs(z) for z in zs) < 3 else
      "PRE-TRENDS PRESENT — the design cannot claim causality as it stands")

fig,ax=plt.subplots(figsize=(8,4))
ax.axhline(0,color="#888",lw=.8); ax.axvline(-0.5,color="#b00",lw=.8,ls="--")
ax.errorbar(ks,bs,yerr=[1.96*(s if np.isfinite(s) else 0) for s in ses],
            fmt="o-",ms=3,lw=1,capsize=2)
ax.set_xlabel("months since first gating"); ax.set_ylabel(PRIMARY)
ax.set_title("Event study · Treatment A")
fig.tight_layout(); fig.savefig(FIG/"nb4_eventstudy_A.png",dpi=150)
print("figure ->", FIG/"nb4_eventstudy_A.png")
plt.show()

## 5 · Power, by randomisation

For each outcome and treatment: draw the real number of treated units at random from the never-treated pool, give them fake adoption months from the real cohort distribution, estimate, repeat. The spread is the standard error; the centre should be zero.

Compare the resulting minimum detectable effects against the prior study's effect — roughly one fewer comment per pull request.

In [ ]:
def fast_placebo_se(y, unit, time, n_treated, cohort_months, rng, n_perm=80, iters=12):
    """Identical maths to placebo_se, ~6x faster:
       (1) Frisch-Waugh - d~ is orthogonal to the FE space, so (d~.y~)==(d~.y)
           and y never needs demeaning inside the loop
       (2) boolean lookup instead of np.isin
       (3) vectorised cohort assignment instead of a per-row dict lookup"""
    nu=int(unit.max())+1; units=np.unique(unit)
    if len(units)<=n_treated or len(cohort_months)==0: return np.nan,np.nan,0
    cm=np.asarray(cohort_months); est=[]
    for _ in range(n_perm):
        chosen=rng.choice(units,size=n_treated,replace=False)
        gvec=np.full(nu,np.iinfo(np.int64).max,dtype=np.int64)
        gvec[chosen]=rng.choice(cm,size=n_treated)
        dd=(time>=gvec[unit]).astype(np.float64)
        if dd.sum()<6: continue
        dt=demean_two_way(dd,unit,time,iters=iters)
        den=float(dt@dt)
        if den<=0: continue
        est.append(float(dt@y)/den)
    if len(est)<20: return np.nan,np.nan,len(est)
    a=np.array(est); return float(a.mean()),float(a.std(ddof=1)),len(a)

N_PERM=80
rng=np.random.default_rng(SEED)
never_mask=(d.g_A>=NEVER).values
rows=[]
for treat in ["A"]+[t for t in TREATS if t!="A"]:
    gcol=d[f"g_{treat}"]
    n_tr=int((gcol<NEVER).groupby(d.repo).max().sum())
    cohorts=np.unique(gcol[gcol<NEVER].values)
    if n_tr<10 or len(cohorts)==0: continue
    cohorts=(cohorts-d.t.min()).astype(int)
    for oc in [PRIMARY,"o2_comments_per_pr","o3_review_coverage",
               "o5_self_merge_rate","o6_direct_push_share","o4_churn_per_merged"]:
        m=never_mask & d[oc].notna().values
        if m.sum()<5000: continue
        u_m=d.u.values[m]; remap={v:i for i,v in enumerate(np.unique(u_m))}
        u_c=np.array([remap[v] for v in u_m],dtype=np.int64)
        c,se,k=fast_placebo_se(d[oc].values[m],u_c,d.tt.values[m].astype(np.int64),
                               n_treated=min(n_tr,len(remap)//2 or 10),
                               cohort_months=cohorts,rng=rng,n_perm=N_PERM)
        mm=d[oc].notna().values
        b,seh,_=twfe_did(d[oc].values[mm],d[f"post_{treat}"].values[mm],
                         d.u.values[mm],d.tt.values[mm])
        sd=float(np.nanstd(d[oc].values))
        rows.append({"treatment":treat,"outcome":oc,"n_treated":n_tr,
                     "placebo_centre":c,"placebo_se":se,"n_draws":k,
                     "mde":mde_from_se(se),"mde_in_sd":mde_from_se(se)/sd if sd else np.nan,
                     "twfe_beta":b,"twfe_se":seh,"outcome_sd":sd})
        print(f"{treat:16s} {oc:22s} n={n_tr:6,}  placebo {c:+.4f}±{se:.4f}  "
              f"MDE {mde_from_se(se):.4f} ({rows[-1]['mde_in_sd']:.3f} sd)  twfe {b:+.4f}",flush=True)
mde=pd.DataFrame(rows)
mde["beta_in_sd"]=mde.twfe_beta/mde.outcome_sd

## 6 · Verdict and export

In [ ]:
bad_placebo = mde[(mde.placebo_se>0) &
                  (mde.placebo_centre.abs() > 2*mde.placebo_se/np.sqrt(mde.n_draws.clip(lower=1)))]
print(f"placebo centres away from zero: {len(bad_placebo)} of {len(mde)}")
if len(bad_placebo): print(bad_placebo[["treatment","outcome","placebo_centre","placebo_se"]].to_string())

prim = mde[(mde.treatment=="A")&(mde.outcome==PRIMARY)]
if len(prim):
    print(f"\nPRIMARY (A -> {PRIMARY}): MDE {prim.mde.iloc[0]:.4f} "
          f"= {prim.mde_in_sd.iloc[0]:.3f} sd of the outcome")
print("\nweakest-powered cells (largest MDE in sd units)")
print(mde.nlargest(8,"mde_in_sd")[["treatment","outcome","n_treated","mde_in_sd"]].to_string(index=False))

out=Path(STATE,"nb4_panel_balanced.parquet")
keep=["repo","t","u","tt"]+OUTCOMES+[c for c in d.columns if c.startswith(("g_","evt_","post_","log_"))]
tmp=out.with_suffix(".parquet.tmp"); d[keep].to_parquet(tmp,index=False); os.replace(tmp,out)
mde.to_csv(Path(STATE,"nb4_mde.csv"),index=False)
summary={"n_rows":int(len(d)),"n_repos":int(d.repo.nunique()),
         "filled_zero_share":float((len(bal)-len(panel))/len(bal)),
         "pre_trend_max_abs_z":float(max(abs(z) for z in zs)),
         "pre_trend_mean":float(np.mean([b for _,b,_ in pre])),
         "primary_mde":float(prim.mde.iloc[0]) if len(prim) else None,
         "primary_mde_sd":float(prim.mde_in_sd.iloc[0]) if len(prim) else None,
         "n_bad_placebo":int(len(bad_placebo)),
         "outcomes":OUTCOMES,"treatments":TREATS}
atomic_write_text(Path(STATE,"nb4_summary.json"),json.dumps(summary,indent=1))
print(json.dumps(summary,indent=1))
print(f"\nwrote {len(d):,} rows -> {out}")